In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry
import numpy as np
import inspect
"""
der Code simuliert die Kirchhoff-Love Gleichung mittels HHJ-methode; der gesamte Rand ist clamped
"""
#region GEOMETRIE
l = 1000
b = 500
t = 5
maxh = 100
order = 2
    
shape = MoveTo(0,0).Rectangle(l,b).Face()
shape.edges.name="dirichlet"
geo = OCCGeometry(shape,dim=2)

mesh = Mesh(geo.GenerateMesh(maxh=maxh))
mesh.Curve(3)
#Draw(mesh)
#endregion

#region MATERIALKONSTANTEN
E  = 70e6      #Glas ~ N'/mm² Elastizitätsmodul - ACHTUNG: N' nicht gleich N!!!      
"""
UMRECHNUNG (m -> mm)

Einheit Pascal: 1 Pa = 1 N/m² 
Einheit Newton: 1 N = 1 kg*m/s²     - dh müssen auch Newton umrechnen

Definiere:
N' := kg*mm/s²      - in Millimeter!!!!
    -> 1 N = 1000 N'

1 Pa = 1 N/m² = 10e-6 N/mm² = 10e-6 * 10e3 N'/mm² = 10e-3 N'/mm²

Glas hat E-Modul von 70e9 Pa (70 GPa) = 70e9 N/m² = 70e9 * 10^(-3) N'/mm² = 70e6 N'/mm²         #neuer Wert
"""
nu = 0.23       #dimensionslos, also bei m und mm gleich
rho = 2.5e-6   #kg/mm³ Dichte 
"""
Dichte von Glas 2500 kg/m³ = 2.5 * 10e3 kg/m³
1kg/m³ = 1000g/10e9mm³ = 10e(-6)g/mm³
-> 2.5 * 10e3 * 10e(-6) g/mm³ = 2.5 * 10e(-3) g/mm³ = 0.0025 g/mm³

"""

g = 9810     # Erdbeschleunigung mm/s²
"""
Erdbeschleunigung g = 9.81 m/s² = 9810 mm/s²
"""   

q = rho * t * g     #Eigengewicht der Platte punktweise!!! - rechte Seite der PDE, also unser f_PDE
#endregion

#region FUNKTIONEN für PDE
Db = E*t**3/(12*(1-nu**2))

def D(A):
    return Db *((1-nu)*A+ nu*Trace(A)*Id(2))

def Dinv(A):
    return 1/Db * (1/(1-nu)*A - nu/(1-nu**2)*Trace(A)*Id(2))
#endregion

# order = 2
#region SIMULATION: FUNKTIONENRAUM,SCHWACHE FORM,VISUALISIERUNG
Q = H1(mesh, order=order+1, dirichlet="dirichlet")      #Durchbiegung
V = HDivDiv(mesh, order=order,dirichlet="dirichlet")    #Momententensor
X = V * Q

(sigma,w),(tau,v)= X.TnT()

n = specialcf.normal(2)

def tang(u):
    return u - (u*n)*n

a = BilinearForm(X,symmetric=True)
a += InnerProduct(Dinv(sigma),tau) * dx
a += div(sigma)*Grad(v) * dx
a += div(tau) * Grad(w) * dx
a += - (sigma[n,:] * tang(Grad(v)) + tau[n,:] * tang(Grad(w))  ) * dx(element_boundary=True)
a.Assemble()

L = LinearForm(X)
L += q * v *dx
L.Assemble()

gf_solution = GridFunction(X)
gf_solution.vec.data = a.mat.Inverse(X.FreeDofs()) * L.vec

gf_sigma, gf_w = gf_solution.components
# Draw(gf_sigma, mesh, name="sigma",deformation=True, scale=0.01, euler_angles=[-60,5,30]) #ist egl ein Tensor, kann nicht klassisch ausgegeben werden
Draw(gf_w, mesh, name="disp",deformation=True, scale = 1000, euler_angles=[-60,5,30])
#endregion


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

BaseWebGuiScene